# Whisper small: baseline vs. fine-tuned, re-scored with the shared evaluation code

Team Tokens · COE025 Project 3 · Finnish ASR

Runs both Whisper systems on the three test sets with `scripts/asr_eval.py`, the evaluation code that
every system in this project uses (same clips, same references, one `normalize()`, corpus-level WER/CER).

| System | Checkpoint | Role |
|---|---|---|
| whisper-small zero-shot | `openai/whisper-small` | baseline (no training by us) |
| whisper-small fine-tuned (ours) | `tia88na/whisper-small-fi-tokens` | contribution (trained by us, `scripts/finnish_asr_whisper_finetune.ipynb`) |

No training happens here. Both systems use the `openai/whisper-small` processor (the one used for fine-tuning).
Decoding is the same as in the fine-tuning notebook: language = Finnish,
task = transcribe, greedy, max 225 new tokens, fp16. The earlier numbers are kept unchanged in `results/legacy/`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (≈ 40–60 min).
At the end `whisper_results.zip` downloads: unzip it into the repo folder (it contains `results/...`).

In [ ]:
!pip -q install -U transformers datasets jiwer accelerate soundfile librosa

In [ ]:
# Get the repo (shared evaluation code + clip lists) and work inside it
import os, sys
if not os.path.isdir("/content/team-tokens-finnish-asr"):
    !git clone -q https://github.com/zehraoz36/team-tokens-finnish-asr /content/team-tokens-finnish-asr
else:
    !git -C /content/team-tokens-finnish-asr pull -q
%cd /content/team-tokens-finnish-asr
sys.path.insert(0, "scripts")
import asr_eval as E
print(E.hardware())

In [ ]:
# Test sets: test_cv, test_parl, test_fleurs (16 kHz). References = `text` column.
test = E.load_test_sets()
for s, d in test.items():
    print(f"{s:12s} {len(d):5d} clips")

## Transcription function

`transcribe_fn(list of 16 kHz float arrays) -> list of raw strings`. The shared code times only this call (RTF).

In [ ]:
import torch, transformers
from transformers import WhisperForConditionalGeneration, WhisperProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
GEN = dict(language="finnish", task="transcribe", num_beams=1, do_sample=False, max_new_tokens=225)
SETTINGS = (f"language=fi, task=transcribe, greedy, max_new_tokens=225, 30 s window, "
            f"transformers {transformers.__version__}")

# Both systems use the processor (log-mel front end + tokenizer) of openai/whisper-small:
# the fine-tuning notebook trained with exactly this processor, and the fine-tuned repo
# stores it only in the newer single-file format (processor_config.json).
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

def make_whisper(model_id):
    model = WhisperForConditionalGeneration.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()
    model.generation_config.forced_decoder_ids = None

    @torch.no_grad()
    def transcribe(arrays):
        feats = processor.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt").input_features
        ids = model.generate(feats.to(DEVICE, DTYPE), **GEN)
        return processor.batch_decode(ids, skip_special_tokens=True)
    return model, transcribe

## Baseline: `openai/whisper-small` zero-shot

In [ ]:
BASELINE = "whisper-small zero-shot"
model, fn = make_whisper("openai/whisper-small")
E.evaluate_system(BASELINE, "openai/whisper-small", fn, test, role="baseline",
                  settings=SETTINGS, precision=str(DTYPE).split(".")[-1], batch_size=16)
del model, fn; torch.cuda.empty_cache()

## Contribution: our fine-tuned model `tia88na/whisper-small-fi-tokens`

In [ ]:
FINETUNED = "whisper-small fine-tuned (ours)"
model, fn = make_whisper("tia88na/whisper-small-fi-tokens")
E.evaluate_system(FINETUNED, "tia88na/whisper-small-fi-tokens", fn, test, role="contribution",
                  settings=SETTINGS + "; trained on train split, best dev-WER checkpoint",
                  precision=str(DTYPE).split(".")[-1], batch_size=16)
del model, fn; torch.cuda.empty_cache()

## Results and comparison with the earlier numbers

The earlier run (`results/legacy/results.csv`) used a slightly different copy of `normalize()` and
normalized some references twice. Small differences (≈ 0.1) are expected; large ones mean something changed.

In [ ]:
import pandas as pd
display(E.show_results())
legacy = pd.read_csv("results/legacy/results.csv").set_index("test_set")
new = E.load_scores().pivot_table(index="test_set", columns="system", values="wer")
print("WER  legacy baseline / new baseline | legacy fine-tuned / new fine-tuned")
for s in E.TEST_SETS:
    print(f"{s:12s} {legacy.loc[s,'baseline_WER']:6.2f} / {new.loc[s,BASELINE]:6.2f} | "
          f"{legacy.loc[s,'finetuned_WER']:6.2f} / {new.loc[s,FINETUNED]:6.2f}")

## Download the outputs

In [ ]:
from google.colab import files
E.export_zip([BASELINE, FINETUNED], "/content/whisper_results.zip")
files.download("/content/whisper_results.zip")